# Fiserv FI Billing — Extraction + Matching Pipeline

This notebook runs in **two independent phases**:

---

### Phase 1 — Extraction
Processes client contract PDFs using GPT vision to extract all dollar-valued line items  
with checkbox states, prices, and section headers.  
**Output:** one Excel per client saved to the `Output` folder.

### Phase 2 — Matching
Reads extraction outputs and matches each item to the Portico dictionary using  
semantic similarity (sentence-transformers + TF-IDF) and GPT-4.1.  
**Output:** one matched Excel per client saved to the `Matched Output` folder.

---

### How to use
1. Update the **CONFIG** cell with your folder paths if needed.
2. Run cells **top to bottom** to load all imports and functions.
3. **Run Phase 1** and wait for all clients to finish extraction.
4. Review outputs in the `Output` folder.
5. **Run Phase 2** to match all extracted clients.

In [1]:
# ── All imports needed for both phases ──────────────────────────
import os
import json
import base64
import math
import time
import re
import numpy as np
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

import fitz  # PyMuPDF
fitz.TOOLS.mupdf_display_errors(False)  # suppress non-fatal MuPDF warnings
import pandas as pd
from PIL import Image
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

print("Imports loaded.")

Imports loaded.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CONFIG  ←  Update these before running
# ═══════════════════════════════════════════════════════════════════

# Base project folder — update this one path if you move the project
BASE_DIR = r"C:\Users\Vedaansh\Desktop\Contract Chatbot - Demo\Existing Scripts"

# Root folder containing one subfolder per client (each with contract PDFs)
CONTRACTS_ROOT =  r"C:\Users\Vedaansh\Desktop\Contract Chatbot - Demo\Existing Scripts\INPUT"

# Where extraction outputs are saved (one Excel per client)
OUTPUT_DIR = OUTPUT_DIR = r"C:\Users\Vedaansh\Desktop\Contract Chatbot - Demo\Output\og_script_output\extraction"

# Where matched outputs are saved (one Excel per client)
MATCHED_OUTPUT_DIR = r"C:\Users\Vedaansh\Desktop\Contract Chatbot - Demo\Output\og_script_output\extraction"

# Portico dictionary — must have 'Description' and 'Material Code' columns on 'Final' sheet
DICTIONARY_FILE =  r"C:\Users\Vedaansh\Desktop\Contract Chatbot - Demo\Input\PORTICO\Portico Dictionary 05-26.xlsx"

# OpenAI API key
API_KEY = "sk-key"

# ── Phase 1 — Extraction settings ───────────────────────────────
EXTRACTION_MODEL = "gpt-5.2-2025-12-11"   # Model used for PDF → dollar item extraction
CHUNK_SIZE       = 12           # PDF pages per API call (reduce if hitting token limits)
DPI              = 600          # Rendering resolution (higher = more accurate, slower)

# ── Phase 2 — Matching settings ─────────────────────────────────
MATCHING_MODEL              = "gpt-4.1-2025-04-14"  # Model for dictionary matching
ITEM_BATCH_SIZE             = 25    # Items per AI matching batch
DICT_CHUNK_SIZE             = 250   # Dictionary entries per chunk sent to AI
MAX_PARALLEL_CALLS          = 22    # Parallel API calls during matching
FUZZY_AUTO_ACCEPT_THRESHOLD = 0.90  # Similarity threshold to auto-accept without AI
SEMANTIC_WEIGHT             = 0.6   # Sentence-transformer similarity weight
LEXICAL_WEIGHT              = 0.4   # TF-IDF similarity weight
USE_CHECKPOINT              = False # Set True to resume an interrupted matching run

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MATCHED_OUTPUT_DIR, exist_ok=True)
print("Config loaded.")

Config loaded.


In [3]:
# ═══════════════════════════════════════════════════════════════════
# SHARED HELPERS  (used by both phases)
# ═══════════════════════════════════════════════════════════════════

def extract_date_from_filename(filename):
    """Parse MM-DD-YYYY date from filename. Returns None if missing or year < 2022."""
    pattern = r'(\d{1,2})-(\d{1,2})-(\d{4})'
    match = re.search(pattern, filename)
    if not match:
        return None
    month, day, year = match.groups()
    date_obj = datetime(int(year), int(month), int(day))
    return None if date_obj.year < 2022 else date_obj


def pdf_to_images(pdf_path, dpi=DPI):
    """Convert each page of a PDF to a PIL Image at the specified DPI."""
    doc = fitz.open(pdf_path)
    pages = []
    for page_index in range(len(doc)):
        page = doc.load_page(page_index)
        mat  = fitz.Matrix(dpi / 72, dpi / 72)
        pix  = page.get_pixmap(matrix=mat)
        img  = Image.open(BytesIO(pix.tobytes("png")))
        pages.append({"page_number": page_index + 1, "image": img})
    return pages


def image_to_base64(image):
    """Encode a PIL Image as a base64 PNG string."""
    buffer = BytesIO()
    image.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")


def is_zero_price(val):
    """Return True if a price value is effectively zero or non-numeric (waived, free, etc.)."""
    if pd.isna(val):
        return False
    s = str(val).strip()
    if s.lower() in ("no charge", "waived", "n/a", "none", "free"):
        return True
    if re.match(r"^over\s*\$", s, re.IGNORECASE):
        return True
    if re.match(r"^\$?\d+(\.\d+)?\s*[MBmb]$", s.strip()):
        return True
    try:
        cleaned = s.replace("$", "").replace(",", "").strip()
        m = re.search(r'-?\d+\.?\d*', cleaned)
        return float(m.group()) == 0 if m else False
    except:
        return False


def clean_price(val):
    """Normalise a price string to $X,XXX.XX format."""
    if pd.isna(val):
        return ""
    s = str(val).strip()
    if s.lower() in ("no charge",) or re.match(r"^over\s*\$", s, re.IGNORECASE):
        return ""
    if re.match(r"^\$?\d+(\.\d+)?\s*[MBmb]$", s.strip()):
        return ""
    negative = bool(re.match(r"^\(", s.replace("$", "").strip()))
    cleaned  = s.replace("$", "").replace(",", "").replace("(", "").replace(")", "").strip()
    m = re.search(r"\d+\.?\d*", cleaned)
    if not m:
        return ""
    value = float(m.group())
    return f"-${value:,.2f}" if negative else f"${value:,.2f}"


def call_api_with_retry(fn, max_retries=5):
    """Call an API function with exponential backoff on rate-limit errors."""
    for attempt in range(max_retries):
        try:
            return fn()
        except Exception as e:
            err = str(e).lower()
            if "429" in str(e) or "rate limit" in err or "too many requests" in err:
                wait = 2 ** attempt
                print(f"  Rate limit hit (attempt {attempt+1}/{max_retries}), retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"API call failed after {max_retries} retries")


print("Shared helpers loaded.")

Shared helpers loaded.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# PHASE 1 FUNCTIONS — Extraction
# ═══════════════════════════════════════════════════════════════════

EXTRACTION_SYSTEM_PROMPT = """
You are a forensic-level contract analysis expert.

Your task is to extract ALL items that have an associated dollar-denominated value.

CRITICAL INSTRUCTIONS FOR CHECKBOX DETECTION:

1. CHECKBOX PLACEMENT: Checkboxes can appear BEFORE or AFTER the fee description. Look carefully at the LEFT side of each line item for empty or filled boxes.

2. CHECKBOX TYPES - Learn to recognize these:
   - CHECKED (fee IS selected): ☑, ☒, ✓, ✗, [X], [x], filled box, box with any mark inside
   - UNCHECKED (fee NOT selected): ☐, □, [ ], empty box, empty square, unfilled checkbox

3. IMPORTANT: An EMPTY BOX (☐ or □) placed before a fee description means the checkbox is UNCHECKED - the fee was NOT selected by the client.

4. HIERARCHICAL STRUCTURE: Some fees have parent-child relationships:
   - Parent item: "☐ VB Website Design and Hosting"
   - Child items indented below: "☐ Up to 5 page site", "☐ 6-10 page site"
   - If the PARENT checkbox is unchecked, all child items under it are also NOT selected.

---------------------------------------------------
ADVANCED CHECKBOX ASSOCIATION LOGIC
---------------------------------------------------

CHECKBOX LOOK-BACK RULE:
For each item with a dollar value:
- If no checkbox appears directly on the same line,
  scan UPWARD visually within the same visual block or section.
- The most recent checkbox appearing immediately before the item
  (within a few lines above) should be considered associated with that item
  UNLESS it clearly belongs to a different section, indentation level, or heading.

VISUAL BLOCK DETECTION:
- Treat items that are vertically grouped together with consistent indentation
  as belonging to the same structural block.
- If a checkbox appears at the start of a block,
  all subsequent indented items under that block inherit that checkbox
  unless they contain their own explicit checkbox.

INDENTATION / HIERARCHY RULES:
- Parent items often:
    • Start at the left margin
    • Contain a checkbox
- Child items often:
    • Are indented to the right
    • Appear directly below a parent
    • May not contain their own checkbox

If a child item does NOT have its own checkbox:
    → It inherits the parent's checkbox state.

If a parent checkbox is EMPTY:
    → All child items are checkbox_checked: false
      unless a child explicitly has its own marked checkbox.

If a parent checkbox is MARKED:
    → Child items are considered true ONLY if visually marked,
      otherwise they inherit the parent's state.

SECTION BOUNDARY RULE:
Do NOT associate a checkbox with an item if:
- A clear section header appears between them
- A large visual spacing break occurs
- A numbering change indicates a new clause

RECENCY PRIORITY RULE:
When multiple checkboxes appear above an item,
associate the item with the closest preceding checkbox
within the same indentation or visual block.

IMPORTANT:
You MUST actively search upward for checkbox association
before concluding checkbox_checked: null.

Failure to associate a visible checkbox in the same block is an error.

---------------------------------------------------
TABLE-LEVEL CHECKBOX ASSOCIATION
---------------------------------------------------

TABLE CONTROL RULE:

If a checkbox appears immediately above a table,
and no intervening section header or clause break exists,
that checkbox is considered the controlling checkbox
for the ENTIRE table.

If such a checkbox exists:
- ALL line items (rows) inside that table inherit the checkbox state
  UNLESS a specific row contains its own explicit checkbox.

ROW-LEVEL OVERRIDE RULE:
If an individual table row contains its own checkbox:
    → That row's checkbox overrides the table-level checkbox.
If no row-level checkbox exists:
    → Use the table-level checkbox state.

MULTI-PAGE TABLE RULE:
If a table spans multiple pages:
    → The checkbox located above the first page of the table
       applies to all continuation pages
       unless a new checkbox explicitly appears above a continued portion.

---------------------------------------------------
CHECKBOX + LABEL + TABLE STRUCTURE RULE
---------------------------------------------------

If a checkbox appears next to a label or option name,
and a table appears directly below that label,
then that checkbox governs the entire table.

A horizontal line, divider line, or table border does NOT break checkbox association.

Only ignore the checkbox if:
- A completely new numbered section begins
- A new major heading appears
- A new clause identifier appears
- A large vertical spacing break clearly separates sections

---------------------------------------------------
NUMBERED SECTION CHECKBOX INHERITANCE RULE
---------------------------------------------------

If a checkbox appears next to a numbered section heading,
then that checkbox governs ALL subordinate content under that section
unless explicitly overridden.

Subordinate content includes:
- Lettered sub-clauses: (a), (b), (c)
- Roman numerals: (i), (ii), (iii)
- Indented fee lines
- Pricing lines that appear below the section header
- Tables that appear under the section

---------------------------------------------------
SECTION HEADER IDENTIFICATION AND INHERITANCE
---------------------------------------------------

You must determine the SECTION HEADER corresponding to each extracted item.

A Section Header is a high-level title that describes the contract section.
These headers typically appear at the TOP of a page, CENTERED, in larger or bold text.

SECTION HEADER ASSOCIATION PROCESS:
STEP 1 — Scan UPWARD from the item to find the nearest section title.
STEP 2 — The FIRST qualifying section header encountered is the correct one.
STEP 3 — If the current page has no header above the item, inherit from the previous page.
STEP 4 — A section header governs all content beneath it until a new header appears.
STEP 5 — All rows in a table under a section header inherit the same header.

Always return the header text EXACTLY as written. Do NOT summarize or rewrite it.

MULTI-LINE HEADER RULE:
Some section headers span multiple lines or consist of a title
and a subtitle directly below it.
If a title line is immediately followed by another descriptive line
with no other content between them, treat the COMBINED text as
the full section header.
Example:
"Fee Exhibit"
"to CheckFree Bill Payment and Delivery Services Schedule"
→ Full header = "Fee Exhibit to CheckFree Bill Payment and Delivery Services Schedule"

Do NOT stop at the first line if a continuation line immediately follows.
Combine all consecutive header lines into one complete section header string.

---------------------------------------------------
VISUAL CHECKBOX REFERENCE
---------------------------------------------------

MARKED checkbox: ☒ ☑ [X] — any square containing crossing diagonal lines or a checkmark.
EMPTY checkbox: ☐ □ [ ] — square outline only, nothing inside.

If a square contains two diagonal crossing lines, it is ALWAYS classified as MARKED.

---------------------------------------------------
STRIKETHROUGH HANDLING RULES
---------------------------------------------------

RULE 1 — IGNORE ALL STRIKETHROUGH VALUES:
If a dollar value appears with a visible strikethrough (the number is crossed out),
you MUST ignore that value entirely.
Do NOT extract it, do NOT include it in the price field, and do NOT reference it in the item description.

RULE 2 — CAPTURE REPLACEMENT PRICE ONLY:
If a strikethrough price is followed by a new non-struck price on the same line,
capture ONLY the new replacement price.
Example: $250 $200 → extract $200 only.

RULE 3 — STRIKETHROUGH + NON-BILLABLE LABEL:
If a dollar value has a strikethrough AND the text next to it is any non-billable label
(including but not limited to: "Included", "Waived", "N/A", "Complimentary", "No Charge", "Free"),
extract the item with the non-billable label as the price (e.g. price = "Included", price = "Waived").

RULE 4 — PARTIAL STRIKETHROUGH:
If only part of a price string is struck through, treat the entire original value as struck
and apply the same rules above.

---------------------------------------------------
EXCLUSION RULES
---------------------------------------------------

RULE — EXCLUDE TOTALS AND SUBTOTALS:
Do NOT extract rows that are labeled as totals, subtotals, or summaries.
This includes lines explicitly prefixed or suffixed with words such as:
"Total", "Subtotal", "Grand Total", "Total One Time Fees", "Total Monthly Fees", etc.
These are aggregations of other line items, not standalone fee items.

RULE — EXCLUDE NON-FEE MONETARY FIGURES:
Do NOT extract dollar values that describe a client's asset size, account balance,
financial metric, or threshold trigger — even if they contain a "$" symbol.
These are contextual figures, not fees or charges.
Examples of values to EXCLUDE:
"total assets up to $73M"
"current assets confirmed to be up to $73M"
"assets over $500M"
A valid fee item must represent a charge, cost, or billable amount payable under the contract.

RULE — EXCLUDE CONDITIONAL THRESHOLD/TRIGGER VALUES:
Do NOT extract dollar values that appear as a condition trigger or threshold label
rather than a fee amount.
Specifically, if a table column is labeled "Asset Size", "Threshold", "Tier", "Over X", or similar,
and the value in that column is a dollar or asset amount describing WHEN a fee activates
(e.g., "Over $100M"), do NOT extract that value.
Only extract the corresponding fee amount from the "Fee" or "Additional Monthly Service Fees"
column in the same row.

RULE — EXCLUDE THRESHOLD AND CONDITION REFERENCES IN PROSE:
Do NOT extract dollar values that appear inside descriptive or legal paragraph text as conditions,
eligibility thresholds, or triggers — even if they contain "$" or "USD".
These are not fees; they describe circumstances under which something applies.
Indicators that a value is a threshold/condition reference include:
"more than $X", "greater than $X", "exceeds $X", "over $X", "above $X"
"holds more than $X in assets", "threshold of $X", "asset size exceeds $X"
The sentence describes WHO qualifies or WHEN something applies, not WHAT is charged.
A valid fee item must be a charge, cost, or billable amount — not a qualifying condition.

Definition:
An item is any service, fee, charge, product, obligation, or clause
that explicitly has a "$" or "USD" amount written.

STEP 1 — Identify EVERY dollar-denominated value. Extract ALL occurrences of "$" or "USD".
STEP 2 — For EACH dollar value, determine the specific item it relates to.
         If the same item has multiple dollar values, extract each as a separate row.

---------------------------------------------------
TABLE COLUMN PRIORITY RULES
---------------------------------------------------

RULE — QTY + UNIT PRICE + AMOUNT COLUMNS:
When a table contains all three of these columns: QTY (or Quantity), UNIT PRICE, and AMOUNT:
- Extract AMOUNT as the price field
- Extract QTY value as the quantity field
- Extract the UNITS value (e.g., "Per Appliance", "Per Hour", "Per Certificate") as the frequency field
- IGNORE the UNIT PRICE column entirely
- Do NOT create separate rows for unit price and amount — extract ONE row per line item only

RULE — AMOUNT IS TBD OR MISSING:
If the AMOUNT column exists but contains "TBD", is blank, or is otherwise not a dollar value:
- Fall back to UNIT PRICE as the price field
- Still extract QTY as quantity and UNITS as frequency
- Still extract ONE row per line item only

RULE — NO AMOUNT COLUMN:
If the table does NOT have an AMOUNT column and only has a single price column
(labeled "Fee", "Price", "Unit Price", "Rate", etc.):
- Extract that column's value as the price field
- Apply normal quantity and frequency extraction rules


Return STRICT JSON:

{
  "items": [
    {
      "item": "<clear description>",
      "checkbox_checked": true or false,
      "price": "<dollar value exactly as written>",
      "page": <page number>,
      "explanation": "brief explanation",
      "section_header": "<section title or null>"
    }
  ]
}

IMPORTANT RULES:
- EMPTY box before the fee → checkbox_checked: false
- MARKED box before the fee → checkbox_checked: true
- No checkbox at all → checkbox_checked: null
- Use ONLY visible text in the images. Do NOT infer missing values.
- Ignore non-dollar currencies, dates, and section numbers.
- Be exhaustive. Missing even one dollar value is an error.

Do not include markdown. Do not include explanations.
"""


def extract_chunk(client, page_chunk, chunk_index):
    """Send one chunk of PDF pages to the extraction model and return parsed items."""
    input_content = []

    # Load the marked checkbox reference image for visual calibration
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd()
    ref_img_path = os.path.join(SCRIPT_DIR, "marked_checkbox_example.png")
    with open(ref_img_path, "rb") as f:
        ref_b64 = base64.b64encode(f.read()).decode("utf-8")

    input_content.append({
        "type": "input_text",
        "text": "The first image is a REFERENCE EXAMPLE of a MARKED checkbox. "
                "Any square containing visible crossing diagonal lines is ALWAYS classified as MARKED."
    })
    input_content.append({"type": "input_image", "image_url": f"data:image/png;base64,{ref_b64}"})
    input_content.append({
        "type": "input_text",
        "text": "The following images are sequential pages from a contract. "
                "Use the checkbox reference example above to correctly classify checkboxes."
    })

    for page in page_chunk:
        image_b64 = image_to_base64(page["image"])
        input_content.append({"type": "input_image", "image_url": f"data:image/png;base64,{image_b64}"})

    response = client.responses.create(
        model=EXTRACTION_MODEL,
        temperature=0,
        input=[
            {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
            {"role": "user",   "content": input_content}
        ]
    )

    raw_text = response.output_text
    stripped = raw_text.strip()
    if stripped.startswith("```"):
        stripped = re.sub(r"^```[a-zA-Z]*\n?", "", stripped)
        stripped = re.sub(r"```$", "", stripped).strip()

    try:
        return json.loads(stripped)
    except Exception as e:
        print(f"  ⚠ JSON parsing failed in chunk {chunk_index}: {e}")
        print(f"  Raw response preview: {raw_text[:300]}")
        return {"items": []}


def extract_items_with_dollar_values(pdf_path):
    """Convert a PDF to images and extract all dollar-valued items via the model."""
    client = OpenAI(api_key=API_KEY)

    print("  Converting PDF to images...")
    pages = pdf_to_images(pdf_path)
    total_pages  = len(pages)
    total_chunks = math.ceil(total_pages / CHUNK_SIZE)
    print(f"  Total pages: {total_pages} | Chunks: {total_chunks}")

    all_items = []
    for chunk_index in range(total_chunks):
        start      = chunk_index * CHUNK_SIZE
        page_chunk = pages[start:start + CHUNK_SIZE]
        print(f"  Processing chunk {chunk_index + 1}/{total_chunks}...")
        try:
            parsed = extract_chunk(client, page_chunk, chunk_index + 1)
        except Exception as e:
            print(f"  ⚠ Chunk {chunk_index + 1} failed: {e} — skipping")
            continue
        for item in parsed.get("items", []):
            all_items.append({
                "Item":            item.get("item"),
                "Price":           item.get("price"),
                "Page":            item.get("page"),
                "Checkbox_Checked": item.get("checkbox_checked"),
                "Explanation":     item.get("explanation"),
                "Section_Header":  item.get("section_header")
            })
    return pd.DataFrame(all_items)


def process_pdf_folder(folder_path):
    """Apply PDF filtering logic and run extraction for all valid PDFs in a client folder."""

    def _year_ok(f):
        return extract_date_from_filename(f) is not None

    def _extract_date_str(f):
        m = re.search(r'\d{1,2}-\d{1,2}-\d{4}', f)
        return m.group(0) if m else f

    def _page_count(f):
        try:
            doc = fitz.open(os.path.join(folder_path, f))
            n   = len(doc)
            doc.close()
            return n
        except Exception as e:
            print(f"  ⚠ Could not open {f} for page count: {e} — treating as 0 pages")
            return 0

    def _is_suffix_duplicate(f, all_files_set):
        """Drop file if removing _SUFFIX from stem yields another file that exists in the folder."""
        stem = f[:-4]
        m    = re.search(r'^(.+)_([^_]+)$', stem)
        if not m:
            return False
        return (m.group(1) + '.pdf') in all_files_set

    # Step 1: collect all valid PDFs (with date >= 2022)
    all_pdfs_raw = sorted([
        f for f in os.listdir(folder_path)
        if f.lower().endswith(".pdf") and _year_ok(f)
    ])

    # Step 2: drop repeated-suffix duplicates (e.g. file_3185055_3185055.pdf)
    all_files_set  = set(all_pdfs_raw)
    suffix_dropped = [f for f in all_pdfs_raw if _is_suffix_duplicate(f, all_files_set)]
    if suffix_dropped:
        print(f"  Suffix duplicates dropped: {len(suffix_dropped)}")
        for f in suffix_dropped:
            print(f"    - {f}")
    all_pdfs = [f for f in all_pdfs_raw if f not in suffix_dropped]

    # Step 3: split into small (<=1MB) and large (>1MB)
    small_files, large_files = [], []
    for f in all_pdfs:
        size = os.path.getsize(os.path.join(folder_path, f))
        (large_files if size > 1 * 1024 * 1024 else small_files).append(f)

    print(f"  {len(small_files)} small file(s), {len(large_files)} large file(s) found.")

    # Step 4: for large files, group by date and keep the one with most pages
    date_groups = {}
    for f in large_files:
        date_groups.setdefault(_extract_date_str(f), []).append(f)
    selected_large = [max(group, key=_page_count) for group in date_groups.values()]
    if len(selected_large) < len(large_files):
        print(f"  Large file dedup: kept {len(selected_large)} of {len(large_files)} large file(s).")

    # Step 5: combine and sort descending by date
    filtered_files = sorted(small_files + selected_large)
    pdf_files = [(f, extract_date_from_filename(f)) for f in filtered_files]
    pdf_files.sort(key=lambda x: x[1], reverse=True)

    # Step 6: extract each file
    all_dataframes = []
    for file, date_obj in pdf_files:
        full_path = os.path.join(folder_path, file)
        print(f"\n  Extracting: {file}")
        try:
            df = extract_items_with_dollar_values(full_path)
        except Exception as e:
            print(f"  ⚠ Failed: {e} — skipping")
            continue
        if df.empty:
            print("  No items extracted.")
            continue
        df["Source Contract"] = file
        df["Date"]            = date_obj.strftime("%Y-%m-%d")
        all_dataframes.append(df)

    return pd.concat(all_dataframes, ignore_index=True) if all_dataframes else pd.DataFrame()


def export_to_excel(df, output_path):
    """Save the extraction DataFrame to an Excel file."""
    if df.empty:
        print("  No dollar values found — skipping export.")
        return
    with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
        df.to_excel(writer, index=False, sheet_name="Dollar_Items")
    print(f"  Extraction saved: {output_path}")


print("Phase 1 functions loaded.")

Phase 1 functions loaded.


---
## Phase 1 — Run Extraction
Run this cell to extract dollar-valued items from all client contract PDFs.  
Already-extracted clients (output Excel exists) are automatically skipped.  
Once complete, review outputs in the `Output` folder before running Phase 2.

In [5]:
# ── PHASE 1: EXTRACTION ─────────────────────────────────────────

client_folders = sorted([
    f for f in os.listdir(CONTRACTS_ROOT)
    if os.path.isdir(os.path.join(CONTRACTS_ROOT, f))
])
print(f"Found {len(client_folders)} client folder(s).")

for client_name in client_folders:
    folder_path  = os.path.join(CONTRACTS_ROOT, client_name)
    output_excel = os.path.join(OUTPUT_DIR, f"Tool Extraction {client_name} Final.xlsx")

    print(f"\n{'='*60}")
    print(f"Client: {client_name}")
    print(f"{'='*60}")

    if os.path.exists(output_excel):
        print(f"  Skipping — output already exists: {os.path.basename(output_excel)}")
        continue

    try:
        df = process_pdf_folder(folder_path)
        export_to_excel(df, output_excel)
    except Exception as e:
        print(f"  ⚠ ERROR: {e}")
        continue

print("\nPhase 1 complete.")

Found 1 client folder(s).

Client: EDUCATIONAL FCU
  Suffix duplicates dropped: 1
    - EDUCATIONAL---GOVERNMENTAL-EMPLOYEES-FEDERAL-CREDIT-UNION---Amendment---4-5-2024---2989558_2989558.pdf
  9 small file(s), 8 large file(s) found.
  Large file dedup: kept 5 of 8 large file(s).

  Extracting: EDUCATIONAL---GOVERNMENTAL-EMPLOYEES-FEDERAL-CREDIT-UNION---Amendment---6-26-2025---3430096.pdf
  Converting PDF to images...
  Total pages: 7 | Chunks: 1
  Processing chunk 1/1...

  Extracting: EDUCATIONAL---GOVERNMENTAL-EMPLOYEES-FEDERAL-CREDIT-UNION---Amendment---6-26-2025---3485557.pdf
  Converting PDF to images...
  Total pages: 7 | Chunks: 1
  Processing chunk 1/1...

  Extracting: EDUCATIONAL---GOVERNMENTAL-EMPLOYEES-FEDERAL-CREDIT-UNION---Amendment---4-24-2025---3414722.pdf
  Converting PDF to images...
  Total pages: 4 | Chunks: 1
  Processing chunk 1/1...
  No items extracted.

  Extracting: EDUCATIONAL---GOVERNMENTAL-EMPLOYEES-FEDERAL-CREDIT-UNION---Amendment---2-20-2025---3337024.pdf

In [6]:
# ═══════════════════════════════════════════════════════════════════
# PHASE 2 FUNCTIONS — Matching
# ═══════════════════════════════════════════════════════════════════

MATCHING_SYSTEM_PROMPT = (
    "You are an expert semantic matching engine.\n"
    "Match each Item to the closest description from the provided dictionary.\n"
    "Rules:\n"
    "- Match on MEANING only. Choose the most specific match.\n"
    "- Only return descriptions VERBATIM from the dictionary.\n"
    "- Monthly fees and one-time/setup fees are NOT interchangeable.\n"
    "- Product specificity beats generic category matches.\n"
    "- VB = Virtual Branch, VBN = Virtual Branch Next (treat as identical).\n"
    "- Return the item EXACTLY as given, character-for-character.\n"
    "- Return STRICT JSON: {\"matches\":[{\"item\":\"...\",\"matched_description\":\"...\",\"confidence_percentage\":95}]}\n"
    "- No markdown."
)


def match_batch(ai_client, items_batch, dict_chunk):
    """Send a batch of items + dictionary chunk to the matching model and return matches."""
    prompt = (
        "Items (copy exactly):\n" + json.dumps(items_batch, indent=2) +
        "\n\nDictionary:\n" + json.dumps(dict_chunk, indent=2) +
        "\n\nFor each item pick the BEST matching description. Return JSON only."
    )
    try:
        response = call_api_with_retry(lambda: ai_client.responses.create(
            model=MATCHING_MODEL, temperature=0,
            input=[
                {"role": "system", "content": MATCHING_SYSTEM_PROMPT},
                {"role": "user",   "content": prompt}
            ]
        ))
        raw = response.output_text.strip()
        if raw.startswith("```"):
            raw = re.sub(r"^```[a-zA-Z]*\n?", "", raw)
            raw = re.sub(r"```$", "", raw).strip()
        return json.loads(raw)["matches"]
    except Exception as e:
        print(f"  WARNING: match_batch failed: {e}")
        return []


def process_client(input_file, dictionary_df, descriptions, lookup_dict, client_name, st_model):
    """Run the full matching pipeline for one client and save matched output Excel."""

    checkpoint_file = os.path.join(MATCHED_OUTPUT_DIR, f"{client_name}_checkpoint.json")
    output_file     = os.path.join(MATCHED_OUTPUT_DIR, f"Matched_{client_name}.xlsx")

    print(f"  Loading: {os.path.basename(input_file)}")
    items_df = pd.read_excel(input_file, sheet_name="Dollar_Items")

    # Filter: keep rows where checkbox is null or checked, and price is non-zero
    checkbox_mask = items_df["Checkbox_Checked"].isna() | (items_df["Checkbox_Checked"] == 1)
    nonzero_mask  = ~items_df["Price"].apply(is_zero_price)
    filtered_df   = items_df[checkbox_mask & nonzero_mask]
    all_items     = filtered_df["Item"].dropna().unique().tolist()
    print(f"  Total rows: {len(items_df)} | To match: {len(filtered_df)} | Unique items: {len(all_items)}")

    # Load checkpoint if resuming
    if USE_CHECKPOINT and os.path.exists(checkpoint_file):
        with open(checkpoint_file) as f:
            ckpt = json.load(f)
        best_matches      = ckpt.get("best_matches", {})
        confidence_scores = ckpt.get("confidence_scores", {})
        print(f"  Checkpoint loaded: {len(best_matches)} items already matched.")
    else:
        best_matches, confidence_scores = {}, {}

    def save_checkpoint():
        with open(checkpoint_file, "w") as f:
            json.dump({"best_matches": best_matches, "confidence_scores": confidence_scores}, f)

    items = [i for i in all_items if i not in best_matches]
    print(f"  Items remaining to match: {len(items)}")

    # ── Stage 1: Fuzzy / semantic matching ───────────────────────
    items_for_ai = []
    if items:
        _LEGAL_NOISE = re.compile(
            r'\b(pursuant to|as defined in|hereinafter|section|schedule|exhibit'
            r'|agreement|contract|dated|effective|the foregoing|referred to as'
            r'|including but not limited to|subject to|in accordance with)\b',
            flags=re.IGNORECASE
        )
        def clean(text):
            return re.sub(r'\s+', ' ', _LEGAL_NOISE.sub(' ', text)).strip()

        clean_items = [clean(s) for s in items]
        clean_desc  = [clean(s) for s in descriptions]

        emb_items = st_model.encode(clean_items, batch_size=64, show_progress_bar=False, normalize_embeddings=True)
        emb_desc  = st_model.encode(clean_desc,  batch_size=64, show_progress_bar=False, normalize_embeddings=True)
        sem_sim   = emb_items @ emb_desc.T

        tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), min_df=1, sublinear_tf=True)
        tfidf.fit(clean_items + clean_desc)
        lex_sim  = (normalize(tfidf.transform(clean_items)) @ normalize(tfidf.transform(clean_desc)).T).toarray()
        combined = SEMANTIC_WEIGHT * sem_sim + LEXICAL_WEIGHT * lex_sim

        _MONTHLY = re.compile(r'monthly', re.IGNORECASE)
        _ONETIME = re.compile(r'(one.?time|setup|set.?up|implementation)', re.IGNORECASE)
        def fee_conflict(a, b):
            return (bool(_MONTHLY.search(a)) and bool(_ONETIME.search(b))) or \
                   (bool(_ONETIME.search(a)) and bool(_MONTHLY.search(b)))

        for i, item in enumerate(items):
            best_j = int(np.argmax(combined[i]))
            score  = float(combined[i, best_j])
            if score >= FUZZY_AUTO_ACCEPT_THRESHOLD and not fee_conflict(item, descriptions[best_j]):
                best_matches[item]      = descriptions[best_j]
                confidence_scores[item] = round(score * 100)
            else:
                items_for_ai.append(item)

        save_checkpoint()
        print(f"  Fuzzy accepted: {len(best_matches)} | Sending to AI: {len(items_for_ai)}")

    # ── Stage 2: AI matching (GPT-4.1) ───────────────────────────
    if items_for_ai:
        ai_client    = OpenAI(api_key=API_KEY)
        item_batches = [items_for_ai[i:i + ITEM_BATCH_SIZE] for i in range(0, len(items_for_ai), ITEM_BATCH_SIZE)]
        dict_chunks  = [descriptions[i:i + DICT_CHUNK_SIZE] for i in range(0, len(descriptions), DICT_CHUNK_SIZE)]

        for batch in item_batches:
            print(f"  AI batch: {len(batch)} items x {len(dict_chunks)} dictionary chunks...")
            candidates = {item: [] for item in batch}

            with ThreadPoolExecutor(max_workers=MAX_PARALLEL_CALLS) as executor:
                futures = {executor.submit(match_batch, ai_client, batch, chunk): chunk for chunk in dict_chunks}
                for future in as_completed(futures):
                    for m in future.result():
                        item = m.get("item")
                        desc = m.get("matched_description")
                        if item in candidates:
                            if desc and desc not in candidates[item]:
                                candidates[item].append(desc)
                        else:
                            for orig in candidates:
                                if item and item.strip().lower() == orig.strip().lower():
                                    if desc and desc not in candidates[orig]:
                                        candidates[orig].append(desc)
                                    break

            non_empty = {k: v for k, v in candidates.items() if v}
            if not non_empty:
                print("  WARNING: No candidates found — skipping ranking.")
                save_checkpoint()
                continue

            # Ranking pass — pick the best candidate per item
            ranking_prompt = (
                "For each item pick the BEST semantic match from its candidates.\n"
                "Prefer specific product matches. Assign CONFIDENCE (0-100).\n"
                f"DATA:\n{json.dumps(non_empty, indent=2)}\n"
                'Return JSON: {"final_matches":[{"item":"...","matched_description":"...","confidence_percentage":95}]}'
            )
            try:
                response = call_api_with_retry(lambda: ai_client.responses.create(
                    model=MATCHING_MODEL, temperature=0,
                    input=[
                        {"role": "system", "content": MATCHING_SYSTEM_PROMPT},
                        {"role": "user",   "content": ranking_prompt}
                    ]
                ))
                raw = response.output_text.strip()
                if raw.startswith("```"):
                    raw = re.sub(r"^```[a-zA-Z]*\n?", "", raw)
                    raw = re.sub(r"```$", "", raw).strip()
                for r in json.loads(raw)["final_matches"]:
                    best_matches[r["item"]]      = r["matched_description"]
                    confidence_scores[r["item"]] = r["confidence_percentage"]
            except Exception as e:
                print(f"  WARNING: Ranking failed: {e}")

            save_checkpoint()

    # ── Merge results and export ──────────────────────────────────
    items_df["Matched Description"]   = items_df["Item"].map(best_matches)
    items_df["Confidence Percentage"] = items_df["Item"].map(confidence_scores)

    # Keep only checked/null rows for export (drop explicitly unchecked)
    export_df = items_df[~items_df["Checkbox_Checked"].isin([0, 0.0, False, "False"])].copy()
    export_df["Checkbox_Checked"] = export_df["Checkbox_Checked"].replace({1: "True", 1.0: "True"})

    # Insert cleaned price column next to raw price
    price_idx = export_df.columns.get_loc("Price")
    export_df.insert(price_idx + 1, "Cleaned Price", export_df["Price"].apply(clean_price))

    # Map matched description to material code
    export_df["Material Code"] = (
        export_df["Matched Description"].str.strip().str.lower().map(lookup_dict)
    )
    export_df = export_df[export_df["Material Code"].notna()]
    export_df.to_excel(output_file, index=False)

    # Clean up checkpoint file on success
    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)

    print(f"  Matched output saved: {output_file}  ({len(export_df)} rows)")
    return len(export_df)


print("Phase 2 functions loaded.")

Phase 2 functions loaded.


---
## Phase 2 — Run Matching
Run this cell **after Phase 1 is complete** and you have reviewed the extraction outputs.  
The dictionary and sentence transformer model load once and are reused across all clients.  
Matching uses GPT-4.1 for items that don't meet the fuzzy similarity threshold.

In [7]:
# ── PHASE 2: MATCHING ────────────────────────────────────────────

# Load dictionary (once for all clients)
print("Loading dictionary...")
dictionary_df = pd.read_excel(DICTIONARY_FILE, sheet_name="Final")
descriptions  = dictionary_df["Description"].dropna().tolist()
lookup_dict   = {
    str(desc).strip().lower(): code
    for desc, code in zip(dictionary_df["Description"], dictionary_df["Material Code"])
    if pd.notna(desc)
}
print(f"Dictionary loaded: {len(descriptions)} entries.")

# Load sentence transformer model (once for all clients)
print("Loading sentence transformer model...")
st_model = SentenceTransformer("all-mpnet-base-v2")
print("Model loaded.\n")

# Find all extraction output files
input_files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.lower().endswith(".xlsx")])
print(f"Found {len(input_files)} extraction file(s) to match.\n")

succeeded, failed, skipped = [], [], []

for fname in input_files:
    # Derive client name from filename (strip prefix/suffix added during extraction)
    client_name = os.path.splitext(fname)[0]
    client_name = re.sub(r'^Tool Extraction\s*', '', client_name, flags=re.IGNORECASE).strip()
    client_name = re.sub(r'\s*Final$', '', client_name, flags=re.IGNORECASE).strip()
    input_path  = os.path.join(OUTPUT_DIR, fname)

    print(f"\n{'='*60}")
    print(f"Client: {client_name}")
    print(f"{'='*60}")

    try:
        n = process_client(input_path, dictionary_df, descriptions, lookup_dict, client_name, st_model)
        (skipped if n == 0 else succeeded).append(client_name)
    except Exception as e:
        print(f"  ERROR: {e}")
        failed.append(client_name)

print(f"\n{'='*60}")
print(f"Phase 2 complete — {len(succeeded)} succeeded | {len(skipped)} skipped | {len(failed)} failed")
if failed:
    print(f"Failed clients: {failed}")

Loading dictionary...
Dictionary loaded: 4387 entries.
Loading sentence transformer model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded.

Found 1 extraction file(s) to match.


Client: EDUCATIONAL FCU
  Loading: Tool Extraction EDUCATIONAL FCU Final.xlsx
  Total rows: 246 | To match: 206 | Unique items: 199
  Items remaining to match: 199
  Fuzzy accepted: 6 | Sending to AI: 193
  AI batch: 25 items x 18 dictionary chunks...
  AI batch: 25 items x 18 dictionary chunks...
  AI batch: 25 items x 18 dictionary chunks...
  AI batch: 25 items x 18 dictionary chunks...
  AI batch: 25 items x 18 dictionary chunks...
  AI batch: 25 items x 18 dictionary chunks...
  AI batch: 25 items x 18 dictionary chunks...
  AI batch: 18 items x 18 dictionary chunks...
  Matched output saved: C:\Users\Vedaansh\Desktop\Contract Chatbot - Demo\Output\og_script_output\extraction\Matched_EDUCATIONAL FCU.xlsx  (114 rows)

Phase 2 complete — 1 succeeded | 0 skipped | 0 failed
